# Universal Fake Image Detector — Inference Notebook

**Derived from** [WisconsinAIVision/UniversalFakeDetect](https://github.com/WisconsinAIVision/UniversalFakeDetect),
the official implementation of the CVPR 2023 paper:

> Ojha, U., Li, Y., & Lee, Y. J. (2023).
> *Towards Universal Fake Image Detectors that Generalize Across Generative Models.* CVPR 2023.
> ([arXiv:2302.10174](https://arxiv.org/abs/2302.10174))

**What this notebook does.** It provides a self-contained, environment-agnostic (CPU/GPU auto-detected)
inference demo of the paper's main model:

- A **frozen CLIP ViT-L/14** image encoder (never trained for fake detection), and
- a **single linear layer** (`nn.Linear(768, 1)`) trained on ProGAN images to separate real from fake.

The resulting score, after a sigmoid, is interpreted as the probability that an image is
machine-generated. The model generalizes across generator families (GANs, diffusion models, ...)
because it relies on generic CLIP feature-space cues rather than generator-specific artifacts.

**Not included here:** training and full benchmark reproduction. Refer to the original repository
for `train.py`, `validate.py`, and the associated datasets.

## 1. Setup

Install the required libraries. The only non-trivial dependency is OpenAI's CLIP package,
which bundles the pretrained ViT-L/14 weights (downloaded automatically on first use).

In [ ]:
%pip install -q torch torchvision ftfy regex tqdm
%pip install -q git+https://github.com/openai/CLIP.git

## 2. Imports and environment detection

The original code assumes CUDA (`img.cuda()` in `validate.py`). This notebook is
environment-agnostic: it selects CUDA when available and falls back to CPU otherwise.

In [ ]:
import os
import urllib.request

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import numpy as np

import clip  # from the openai/CLIP package

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cpu":
    print("No GPU detected; inference will work but run noticeably slower.")

# Reproducibility (mirrors SEED handling in the original validate.py)
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED);

## 3. Load the pretrained detector

### 3.1 The linear head

The paper's main checkpoint, `fc_weights.pth`, contains the trained linear layer placed on top of
the frozen CLIP ViT-L/14 encoder (feature dimension 768, exactly as in `models/clip_models.py`
of the source repository, where `CHANNELS = {"ViT-L/14": 768}`).

The official repository distributes the weights via the project page's Google Drive link
(see its README). For convenience, this notebook fetches the identical file from a public
Hugging Face mirror. If the mirror is unavailable, download `fc_weights.pth` manually from the
[project page](https://utkarshojha.github.io/universal-fake-detection/) and place it in the
`pretrained_weights` directory.

In [ ]:
WEIGHTS_URL = (
    "https://huggingface.co/siddharthksah/deepsafe-weights/resolve/main/"
    "universalfakedetect/fc_weights.pth"
)

os.makedirs("pretrained_weights", exist_ok=True)
CKPT_PATH = "pretrained_weights/fc_weights.pth"

if not os.path.exists(CKPT_PATH):
    print(f"Downloading fc_weights.pth ...")
    urllib.request.urlretrieve(WEIGHTS_URL, CKPT_PATH)
print(f"Checkpoint ready: {CKPT_PATH} ({os.path.getsize(CKPT_PATH)} bytes)")

### 3.2 Rebuild the model exactly as in the source repository

`models/clip_models.py` defines the detector as a frozen CLIP model plus a linear layer. Only the linear layer's parameters come from the checkpoint; the CLIP encoder weights are the standard public ViT-L/14 weights.

In [ ]:
class CLIPFakeDetector(nn.Module):
    """Faithful re-implementation of models/clip_models.py::CLIPModel for ViT-L/14.

    features = clip.encode_image(x);  logits = fc(features)
    """

    CHANNELS = {"RN50": 1024, "ViT-L/14": 768}

    def __init__(self, name="ViT-L/14", num_classes=1):
        super().__init__()
        self.model, _ = clip.load(name, device="cpu")  # frozen encoder
        self.fc = nn.Linear(self.CHANNELS[name], num_classes)

    def forward(self, x, return_feature=False):
        features = self.model.encode_image(x)
        if return_feature:
            return features
        return self.fc(features)


detector = CLIPFakeDetector("ViT-L/14")

# Load the trained linear head (the CLIP backbone stays frozen and untouched)
state_dict = torch.load(CKPT_PATH, map_location="cpu")
if "fc.weight" in state_dict:
    detector.fc.load_state_dict({k[3:]: v for k, v in state_dict.items()})
else:
    detector.fc.load_state_dict(state_dict)

detector = detector.to(DEVICE).eval()
print("Detector loaded. Trainable parameters in the head only:",
      sum(p.numel() for p in detector.fc.parameters()))

## 4. Preprocessing

`validate.py` in the source repository uses `CenterCrop(224)` with CLIP normalization statistics.
The evaluation datasets there are 256x256, so the center crop is applied directly. For arbitrary
user images of any size, this notebook first resizes the shorter side to 256 pixels, then applies
the same center crop and normalization, which reproduces the original input distribution as closely
as possible.

In [ ]:
MEAN = (0.48145466, 0.4578275, 0.40821073)   # CLIP statistics, from validate.py
STD  = (0.26862954, 0.26130258, 0.27577711)

transform = transforms.Compose([
    transforms.Resize(256, antialias=True),      # for arbitrary image sizes
    transforms.CenterCrop(224),                  # as in the original pipeline
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

def load_image(path):
    """Open an image file, convert to RGB, and apply the model's preprocessing."""
    img = Image.open(path).convert("RGB")
    return transform(img).unsqueeze(0)

## 5. Inference

`validate.py` applies a sigmoid to the model logit; a probability above 0.5 means FAKE,
and below 0.5 means REAL. The same convention is used here.

In [ ]:
@torch.no_grad()
def predict(paths, batch_size=32):
    """Return a list of dictionaries with the fake-probability per image."""
    results = []
    for i in range(0, len(paths), batch_size):
        chunk = paths[i:i + batch_size]
        batch = torch.cat([load_image(p) for p in chunk]).to(DEVICE)
        logits = detector(batch).flatten()
        probs = logits.sigmoid().cpu().numpy()
        for p, prob in zip(chunk, probs):
            results.append({
                "image": p,
                "p_fake": float(prob),
                "label": "FAKE" if prob > 0.5 else "REAL",
            })
    return results

def show_results(results):
    for r in results:
        print(f"{r['label']:>4}  p(fake)={r['p_fake']:.4f}  {os.path.basename(r['image'])}")

### 5.1 Classify your own images

Place image files (png/jpg/jpeg/bmp) in a directory and set `IMAGE_DIR` below, or list files
explicitly in `IMAGE_PATHS`. The cell then reports the model verdict for each image.

In [ ]:
IMAGE_DIR = "images"      # <-- change me
IMAGE_PATHS = []          # or list explicit files, e.g. ["photo.png", "generated.jpg"]

EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp"}

paths = list(IMAGE_PATHS)
if os.path.isdir(IMAGE_DIR):
    paths += [
        os.path.join(IMAGE_DIR, f)
        for f in sorted(os.listdir(IMAGE_DIR))
        if os.path.splitext(f)[1].lower() in EXTENSIONS
    ]

assert paths, "No images found: set IMAGE_DIR or IMAGE_PATHS."
results = predict(paths)
show_results(results)

## 7. Gradio web interface

This section wraps the detector in a small [Gradio](https://www.gradio.app/) web application,
so that images can be classified interactively from a browser, via upload or webcam capture.

Launch options:

- `SHARE = True`: Gradio creates a temporary public URL (hosted tunnel, valid for 72 hours),
  accessible from any device on the web. Useful for demos on machines without port forwarding,
  such as Colab or a machine behind NAT.
- `SHARE = False`: the interface is served locally at the printed URL (default
  `http://127.0.0.1:7860`), reachable only from the machine running this notebook.

To expose the interface permanently on your own server, run this notebook on a machine with a
public IP or a reverse proxy, and use `demo.launch(server_name="0.0.0.0", server_port=7860)`.

In [ ]:
%pip install -q gradio

In [ ]:
import shutil

import gradio as gr

MEAN = (0.48145466, 0.4578275, 0.40821073)
STD  = (0.26862954, 0.26130258, 0.27577711)

@torch.no_grad()
def classify_pil(img: Image.Image) -> dict:
    """Classify a single PIL image; return Gradio-style label dictionary."""
    if img is None:
        return {"REAL": 0.0, "FAKE": 0.0}
    img = img.convert("RGB")
    x = transform(img).unsqueeze(0).to(DEVICE)
    p_fake = float(detector(x).flatten().sigmoid().cpu())
    return {"REAL": 1.0 - p_fake, "FAKE": p_fake}

@torch.no_grad()
def classify_folder(folder: str) -> list[tuple]:
    """Classify every image in a folder; return Gradio gallery rows (image, caption)."""
    if not folder or not os.path.isdir(folder):
        return []
    EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp"}
    names = sorted(
        f for f in os.listdir(folder)
        if os.path.splitext(f)[1].lower() in EXTENSIONS
    )
    gallery = []
    for name in names:
        path = os.path.join(folder, name)
        img = Image.open(path).convert("RGB")
        p_fake = float(
            detector(transform(img).unsqueeze(0).to(DEVICE)).flatten().sigmoid().cpu()
        )
        verdict = "FAKE" if p_fake > 0.5 else "REAL"
        gallery.append((img, f"{name} — {verdict} (p_fake={p_fake:.4f})"))
    return gallery

@torch.no_grad()
def classify_file_list(files: list) -> list[tuple]:
    """Classify uploaded files; return Gradio gallery rows (image, caption)."""
    gallery = []
    for file in files or []:
        try:
            img = Image.open(file.name if hasattr(file, "name") else file).convert("RGB")
        except Exception:
            continue
        p_fake = float(
            detector(transform(img).unsqueeze(0).to(DEVICE)).flatten().sigmoid().cpu()
        )
        verdict = "FAKE" if p_fake > 0.5 else "REAL"
        gallery.append((img, f"{verdict} (p_fake={p_fake:.4f})"))
    return gallery

In [ ]:
EXAMPLES_DIR = "examples"  # optional: put sample real/fake images here for the demo
if not os.path.isdir(EXAMPLES_DIR):
    os.makedirs(EXAMPLES_DIR, exist_ok=True)

with gr.Blocks(title="Universal Fake Image Detector") as demo:

    gr.Markdown(
        """
        # Universal Fake Image Detector
        CLIP ViT-L/14 + a single linear layer (Ojha, Li & Lee, CVPR 2023).
        A probability above 0.5 for FAKE means the image is predicted to be machine-generated.
        """
    )

    with gr.Tab("Single image"):
        with gr.Row():
            with gr.Column():
                single_in = gr.Image(type="pil", label="Input image", sources=["upload", "webcam"])
                classify_btn = gr.Button("Classify", variant="primary")
            with gr.Column():
                single_out = gr.Label(label="Verdict", num_top_classes=2)

    with gr.Tab("Batch upload"):
        batch_in = gr.File(label="Image files", file_count="multiple")
        batch_btn = gr.Button("Classify all", variant="primary")
        batch_out = gr.Gallery(label="Results", columns=4, height="auto")

    with gr.Tab("Folder on server"):
        folder_in = gr.Textbox(label="Folder path", value="images")
        folder_btn = gr.Button("Classify folder", variant="primary")
        folder_out = gr.Gallery(label="Results", columns=4, height="auto")

    gr.Markdown(
        """
        Note: p(fake) = sigmoid(fc(CLIP_features)). Scores near 0.5 indicate uncertainty.
        Heavy JPEG compression or blur can degrade detection accuracy.
        """
    )

    classify_btn.click(classify_pil, inputs=single_in, outputs=single_out)
    batch_btn.click(classify_file_list, inputs=batch_in, outputs=batch_out)
    folder_btn.click(classify_folder, inputs=folder_in, outputs=folder_out)

print("Interface defined. Launch in the next cell.")

In [ ]:
# Set SHARE = True to obtain a temporary public URL (valid ~72 hours).
# Set SHARE = False for local access only (http://127.0.0.1:7860).
SHARE = True

demo.queue()
demo.launch(share=SHARE)

The launched interface provides three tabs:

1. Single image: upload or capture an image and obtain the verdict with class probabilities.
2. Batch upload: select multiple files and receive a gallery annotated with per-image verdicts.
3. Folder on server: point to a directory on the machine running the notebook and classify its
   entire contents at once.

In Colab, the output cell displays the interface directly. When `SHARE = True`, a second public
URL of the form `https://<random-id>.gradio.live` is printed; it can be shared with anyone for
approximately 72 hours. To stop the interface, interrupt the cell or run `demo.close()`.

## 6. Interpretation and limitations

- The reported value is `p(fake) = sigmoid(fc(ClipFeatures))`, not a calibrated probability.
- The model was trained exclusively on ProGAN images, yet it generalizes to unseen generators
  (BigGAN, CycleGAN, StyleGAN, Glide, LDM, Stable Diffusion, ...), which is the paper's core claim.
- Known weaknesses discussed in the paper include heavy JPEG compression and blur, which erode
  the feature-space cues the linear layer depends on.
- A score near 0.5 should be treated as uncertain rather than as evidence for either class.
- Accuracy is well below 100 percent on every domain: this detector is a strong general-purpose
  signal, not a definitive verdict.

## Citation

```bibtex
@inproceedings{ojha2023fakedetect,
      title={Towards Universal Fake Image Detectors that Generalize Across Generative Models},
      author={Ojha, Utkarsh and Li, Yuheng and Lee, Yong Jae},
      booktitle={CVPR},
      year={2023},
}
```

## Source attribution

This notebook is derived from
[UniversalFakeDetect](https://github.com/WisconsinAIVision/UniversalFakeDetect) (MIT license).
The model definition, preprocessing statistics, and sigmoid decision rule follow
`models/clip_models.py`, `models/__init__.py`, and `validate.py` of that repository.